# 51. Time-series forecasting without future leakage

**Level:** Intermediate to advanced

**Prerequisites:** 11-13: splits and regression.

[Course index](00_START_HERE.ipynb) | [Previous](50_Fairness_Privacy_and_Responsible_AI.ipynb) | [Next](52_Agents_Workflows_and_Bounded_Tools.ipynb)

**Topics:** Time series, Forecasting, Temporal splits, Lag features, TimeSeriesSplit, Baselines.

**Matching script:** `examples\51_Time_Series_and_Forecasting.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Forecasting predicts future observations from information available at the prediction time.
Randomly shuffling temporal rows can leak future regimes or correlated near-duplicates into
training. Use chronological splits, and consider a gap when label windows overlap.

Lag features use earlier measurements. Rolling means must be shifted so the target itself
does not enter its predictors. Imputation and normalization must not look ahead. A one-step
walk-forward evaluation can use newly observed earlier test values, but a fixed-origin
multi-step forecast cannot assume those future observations are already known.

Compare with simple baselines such as the most recent value or a seasonal repeat. A complex
model that cannot beat the appropriate baseline has not demonstrated value. Model drift,
unusual incidents, missing intervals, and irregular timestamps complicate deployment.

## Mathematics or data contract

For target y_t, a lag-one feature is y_(t-1). A persistence baseline predicts y_hat_t=y_(t-1).

## Important limitation

The held-out evaluation is explicitly one-step walk-forward with observed prior values, not a fixed-origin forecast.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Chronological one-step regression with a baseline

**Purpose:** Make time availability explicit.

### Read the code in this order
1. Create a synthetic autoregressive series.
2. Build targets only from prior lag values.
3. Train on the earlier partition and evaluate later observations.
4. Report MAE for both model and persistence without selecting on test.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
one-step test MAE: 0.154
persistence MAE: 0.166
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(42)
series = np.zeros(240)
for index in range(1, len(series)):
    series[index] = 0.8 * series[index-1] + rng.normal(0, 0.2)
X, y = series[:-1, None], series[1:]
cutoff = 170
model = LinearRegression().fit(X[:cutoff], y[:cutoff])
predicted = model.predict(X[cutoff:])
model_error = mean_absolute_error(y[cutoff:], predicted)
baseline_error = mean_absolute_error(y[cutoff:], X[cutoff:, 0])
assert predicted.shape == y[cutoff:].shape
assert model_error < 0.4 and baseline_error < 0.4
print("one-step test MAE:", round(model_error, 3))
print("persistence MAE:", round(baseline_error, 3))

one-step test MAE: 0.154
persistence MAE: 0.166


### Interpret example 1

The result should demonstrate this purpose: **Make time availability explicit.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Implement a fixed-origin ten-step recursive forecast and explain why its uncertainty differs.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Time series` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: For target y_t, a lag-one feature is y_(t-1). A persistence baseline predicts y_hat_t=y_(t-1).
- Name the boundary of the demonstration: The held-out evaluation is explicitly one-step walk-forward with observed prior values, not a fixed-origin forecast.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.